[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/text-classification-practice/02_keras_text/02_keras_text_solutions.ipynb)

# 02. Keras 텍스트 분류 — 연습 문제 해설

[02_keras_text.ipynb](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/text-classification-practice/02_keras_text/02_keras_text.ipynb) 끝의
연습 문제 6개에 대한 정답 코드와 해설입니다. **먼저 직접 시도해본 뒤** 참고하세요.

> **신경망은 실행할 때마다 결과가 조금씩 달라집니다.** seed를 고정해도 환경(CPU/GPU, TensorFlow 버전)에
> 따라 소수점 이하가 달라집니다. 아래 숫자는 방향을 보는 용도이지 정답이 아닙니다.
>
> **문제 6은 학습 데이터를 전부 쓰므로 3분쯤 걸립니다.** 나머지는 문제당 30초~2분입니다.

In [ ]:
import os
import sys
import time

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q pandas scikit-learn matplotlib

YNAT = "https://raw.githubusercontent.com/KLUE-benchmark/KLUE/main/klue_benchmark/ynat-v1.1"

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

RANDOM_STATE = 42

raw = pd.read_json(f"{YNAT}/ynat-v1.1_train.json")
data = raw[["title", "label"]].sample(20_000, random_state=RANDOM_STATE).reset_index(drop=True)

X, y_text = data["title"].values, data["label"].values
X_train, X_valid, y_train_text, y_valid_text = train_test_split(
    X, y_text, test_size=0.2, stratify=y_text, random_state=RANDOM_STATE
)

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_text)
y_valid = label_encoder.transform(y_valid_text)
N_CLASSES = len(label_encoder.classes_)

print("학습", len(X_train), "· 검증", len(X_valid))

본문에서 쓴 두 가지 모델(단어 단위 `average`, 글자 단위 `conv`)을 함수 하나로 묶어둡니다.
문제마다 인자만 바꿔 부릅니다.

In [ ]:
def train_model(split=None, seq_len=12, max_tokens=20_000, embed_dim=64, head="average",
                X_fit=None, y_fit=None, X_ev=None, y_ev=None):
    """(모델, 벡터화 레이어, 검증 정확도, 학습 시간)을 돌려준다.

    split=None      → 공백으로 자르는 단어 단위
    split='character' → 글자 단위
    """
    X_fit = X_train if X_fit is None else X_fit
    y_fit = y_train if y_fit is None else y_fit
    X_ev = X_valid if X_ev is None else X_ev
    y_ev = y_valid if y_ev is None else y_ev

    keras.utils.set_random_seed(RANDOM_STATE)
    kwargs = {} if split is None else {"split": split}

    vectorize = layers.TextVectorization(
        max_tokens=max_tokens, output_sequence_length=seq_len, **kwargs
    )
    vectorize.adapt(X_fit)

    inputs = keras.Input(shape=(1,), dtype=tf.string)
    h = vectorize(inputs)
    h = layers.Embedding(max_tokens, embed_dim)(h)
    if head == "conv":
        h = layers.Conv1D(128, 3, activation="relu")(h)
        h = layers.GlobalMaxPooling1D()(h)
    else:
        h = layers.GlobalAveragePooling1D()(h)
    h = layers.Dropout(0.3)(h)
    h = layers.Dense(64, activation="relu")(h)
    outputs = layers.Dense(N_CLASSES, activation="softmax")(h)

    model = keras.Model(inputs, outputs)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    started = time.time()
    model.fit(
        X_fit, y_fit, validation_data=(X_ev, y_ev),
        epochs=30, batch_size=64, verbose=0,
        callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=3,
                                                 restore_best_weights=True)],
    )
    return model, vectorize, model.evaluate(X_ev, y_ev, verbose=0)[1], time.time() - started


print("준비 완료")

---

## 문제 1. `SEQ_LEN`을 4로 줄이면?

In [ ]:
for seq_len in [4, 12]:
    _, _, acc, _ = train_model(seq_len=seq_len)
    print(f"SEQ_LEN={seq_len:>2}  검증 정확도 {acc:.4f}")

print("\n앞 4단어만 남기면:")
for s in X_train[:4]:
    print("  ", " ".join(s.split()[:4]), "   <- 원본:", s)

**0.7523 → 0.7150. 3.7%p 떨어집니다.**

01번 연습 문제 6번에서 **앞 3단어 0.7825, 뒤 3단어 0.6285** 로 앞쪽이 훨씬 유용하다는 결과가
나왔습니다. 여기서도 앞 4단어만 남겨 0.7150이 나오니 방향은 일치합니다.

**그래도 손실이 작지 않습니다.** 제목이 평균 6.6단어인데 4단어만 남기면 **40%를 버리는 것**입니다.
01번 실험이 말한 것은 "앞쪽이 더 유용하다"였지, **"뒤쪽이 쓸모없다"가 아니었습니다.**
뉴스 제목은 `…` 뒤에 부연이 붙는 구조가 많아
(`美코스트코서 사흘만에 또 총격…용의자 사망·2명 부상`), 뒤쪽에도 신호가 남아 있습니다.

**교훈:** 시퀀스 길이를 줄이면 **뒤가 잘립니다.** 계산을 아끼려고 자르기 전에,
**분위수를 보고 95~99%를 덮는 길이**를 고르세요. 여기서는 12가 그 값이었고,
자를 이유가 애초에 없었습니다(최대 13단어).

---

## 문제 2. `MAX_TOKENS`와 OOV

In [ ]:
for max_tokens in [5_000, 20_000, 44_901]:
    model, vectorize, acc, _ = train_model(max_tokens=max_tokens)
    ids = vectorize(X_valid).numpy()
    oov = (ids == 1).sum() / (ids != 0).sum()
    print(f"max_tokens={max_tokens:>6}  사전 {len(vectorize.get_vocabulary()):>6}"
          f"  OOV {oov:.1%}  정확도 {acc:.4f}  파라미터 {model.count_params():>10,}")

| `max_tokens` | 사전 | OOV | 정확도 | 파라미터 |
|---|---|---|---|---|
| 5,000 | 5,000 | 49.9% | 0.7383 | 324,615 |
| 20,000 | 20,000 | 38.5% | **0.7523** | 1,284,615 |
| 44,901 (전체 어휘) | 44,901 | 31.5% | 0.7492 | 2,878,279 |

**OOV는 계속 줄어드는데 정확도는 20,000에서 정점을 찍고 오히려 내려갑니다.**

이것이 이 문제의 핵심입니다. **OOV를 줄이는 것 자체는 목표가 될 수 없습니다.**

사전을 5,000에서 20,000으로 키우면 이득입니다. 그 구간에 추가되는 단어는 **여러 번 나오는 단어**라
임베딩을 학습할 기회가 있습니다. 그런데 20,000을 넘어서면 사정이 다릅니다. 본문 1절에서 확인했듯
**어휘의 74%가 딱 한 번만 나오는 단어**입니다. 그런 단어를 사전에 넣어봐야,

- 그 단어의 64차원 벡터는 **한 번의 역전파로만** 갱신됩니다. 사실상 초기값 그대로입니다
- 검증 데이터에서 그 단어를 만나면 학습되지 않은 난수 벡터가 입력됩니다.
  차라리 `[UNK]`로 뭉개는 편이 나았습니다
- 파라미터만 2.2배로 늘어 과적합이 심해집니다

**그럼 OOV 38.5%는 어떻게 줄이나요?** 사전을 키워서가 아니라 **토큰 단위를 바꿔서** 줄입니다.
본문 6절에서 글자 단위로 바꾸자 사전이 1,511개로 줄고 OOV가 사실상 사라졌으며,
정확도가 0.7928까지 올라갔습니다. **같은 증상에 대한 올바른 처방이 무엇인지**를 보여주는 대비입니다.

---

## 문제 3. 임베딩 차원 16 / 64 / 256

In [ ]:
for dim in [16, 64, 256]:
    model, _, acc, elapsed = train_model(embed_dim=dim)
    print(f"embed_dim={dim:>3}  정확도 {acc:.4f}  파라미터 {model.count_params():>10,}  {elapsed:.0f}초")

| 차원 | 정확도 | 파라미터 | 학습 시간 |
|---|---|---|---|
| 16 | 0.7315 | 321,543 | 8초 |
| 64 | 0.7523 | 1,284,615 | 32초 |
| 256 | **0.7590** | 5,136,903 | **109초** |

**키울수록 조금씩 좋아지긴 합니다. 그런데 값이 맞지 않습니다.**

64 → 256에서 얻은 것은 **+0.0067**입니다. 대신 파라미터는 4배, 학습 시간은 **3.4배**가 됐습니다.
01번 7절의 기준으로 보면 +0.0067은 분할을 바꿀 때의 흔들림 폭(0.01)보다도 작습니다.
**여러 분할에서 짝지어 재보지 않으면 진짜 개선인지도 알 수 없는 크기**입니다.

임베딩 차원은 **"단어 하나를 몇 개의 숫자로 표현할까"** 입니다. 너무 작으면 서로 다른 단어를
구분할 자리가 부족하고(16이 그 경우입니다), 너무 크면 학습할 값만 늘어납니다.
실무의 대략적인 기준은 어휘가 수천~수만이면 50~300차원입니다.

**그리고 무엇보다,** 본문 6절에서 토큰 단위를 바꿔 얻은 것이 **+0.044**였습니다.
차원을 4배로 키워 얻은 것의 **여섯 배**입니다. 같은 시간을 쓴다면 어디에 써야 하는지 분명합니다.

---

## 문제 4. 임베딩 공간에서 `손흥민`과 가까운 단어

In [ ]:
model, vectorize, acc, _ = train_model()

vocab = [str(w) for w in vectorize.get_vocabulary()]

# model.layers는 [입력, TextVectorization, Embedding, ...] 순서라 2번이 임베딩 층입니다.
# (model.summary()로 순서를 확인할 수 있습니다.) get_weights()[0]이 임베딩 행렬입니다.
embedding = model.layers[2].get_weights()[0][: len(vocab)]

# 코사인 유사도 = 정규화한 벡터의 내적
normalized = embedding / (np.linalg.norm(embedding, axis=1, keepdims=True) + 1e-9)


def 비슷한_단어(word, k=8):
    i = vocab.index(word)
    sim = normalized @ normalized[i]
    return [(vocab[j], round(float(sim[j]), 2)) for j in np.argsort(-sim) if j != i][:k]


for w in ["손흥민", "코스피"]:
    print(f"[{w}]과 가까운 단어")
    for word, score in 비슷한_단어(w):
        print(f"   {word:<12} {score}")
    print()

**의미가 비슷한 단어가 실제로 가깝습니다.**

```
[손흥민]  류현진 0.98 · 추신수 0.98 · 여자배구 0.98 · 프로배구 0.98 · 남자배구 0.97 · 호날두 0.97 ...
[코스피]  코스닥 0.98 · 특징주 0.97 · 영업익 0.97 · 대출 0.97 · 하나금투 0.97 · 실적 0.97 ...
```

모델은 사전에서 이 단어들의 뜻을 배운 적이 없습니다. **"같은 주제를 예측하게 만드는 단어끼리
가까워지도록" 역전파가 벡터를 밀어놓은 것**입니다. 이것이
[임베딩](https://github.com/karzit/temp/blob/master/glossary.md#embedding)이 하는 일입니다.

**여기서 "의미"란 무엇일까요.** 축구 선수 `손흥민`과 야구 선수 `류현진`이 0.98로 붙어 있습니다.
일반적인 의미로는 꽤 다른 두 사람인데, **이 문제에서는 완전히 같습니다** — 둘 다 `스포츠`를
가리키는 신호일 뿐입니다. `여자배구`, `프로농구` 같은 종목 이름과도 구별되지 않습니다.

**즉 여기서의 "의미"는 "주제 분류에 쓸모 있는 정도"입니다.** 이 임베딩으로
"손흥민과 호날두는 같은 종목, 류현진은 다른 종목"을 구분하려 하면 실패합니다.
그런 것까지 아는 벡터를 원한다면 **사전 학습된 임베딩**(Word2Vec, FastText, KLUE-RoBERTa)을 씁니다.
[RAG 실습](https://github.com/karzit/temp/blob/master/notebooks/rag-pipeline-practice/README.md)에서 쓴 문장 임베딩이 그런 경우입니다.

> **한 가지 더.** 유사도가 전부 0.97~0.98로 몰려 있는 것도 눈여겨보세요. 학습이 짧게 끝나서
> (7 epoch) 벡터들이 넓게 퍼질 기회가 없었던 탓입니다. 절댓값보다 **순위**를 보세요.

---

## 문제 5. TF-IDF 모델과 앙상블

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline

# 본문 6절의 글자 단위 conv 모델
nn_model, _, nn_acc, _ = train_model(split="character", seq_len=40, max_tokens=3000, head="conv")

tfidf_model = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)),
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
).fit(X_train, y_train_text)

# 두 모델의 클래스 순서를 맞춰야 확률을 더할 수 있습니다.
proba_tfidf = np.zeros((len(X_valid), N_CLASSES))
for k, cls in enumerate(tfidf_model.classes_):
    proba_tfidf[:, list(label_encoder.classes_).index(cls)] = tfidf_model.predict_proba(X_valid)[:, k]

proba_nn = nn_model.predict(X_valid, verbose=0)
proba_mix = (proba_tfidf + proba_nn) / 2

for name, proba in [("TF-IDF", proba_tfidf), ("신경망", proba_nn), ("앙상블", proba_mix)]:
    print(f"{name:<8} 정확도 {accuracy_score(y_valid, proba.argmax(axis=1)):.4f}")

틀림_tfidf = proba_tfidf.argmax(axis=1) != y_valid
틀림_nn = proba_nn.argmax(axis=1) != y_valid
print(f"\n둘 다 틀림 {int((틀림_tfidf & 틀림_nn).sum())}건"
      f" · TF-IDF만 틀림 {int((틀림_tfidf & ~틀림_nn).sum())}건"
      f" · 신경망만 틀림 {int((틀림_nn & ~틀림_tfidf).sum())}건")

**앙상블이 손해입니다. 0.8455 → 0.8225.**

| | 정확도 |
|---|---|
| TF-IDF | **0.8455** |
| 신경망 (글자 conv) | 0.7927 |
| 앙상블 | 0.8225 |

앙상블은 흔히 "공짜 점심"처럼 소개되지만, 여기서는 **더 좋은 모델을 더 나쁘게 만들었습니다.**
오답 분포를 보면 이유가 분명합니다.

| | 건수 |
|---|---|
| 둘 다 틀림 | 440 |
| TF-IDF만 틀림 | 178 |
| **신경망만 틀림** | **389** |

**신경망만 틀린 것이 TF-IDF만 틀린 것의 두 배가 넘습니다.** 확률을 평균 내면,
TF-IDF가 맞힌 389건 중 일부가 신경망의 틀린 확신에 끌려 넘어갑니다.
반대 방향으로 구제되는 것은 178건뿐이니, 산술적으로 손해입니다.

**앙상블이 이득이려면 조건이 두 가지입니다.**

1. **서로 다른 실수를 할 것** — 이 조건은 만족합니다(둘 다 틀린 440건 외에 567건이 엇갈립니다)
2. **실력이 엇비슷할 것** — 이 조건이 깨졌습니다. 5%p 차이는 너무 큽니다

**가중치를 주면 어떨까요?** `0.7 × TF-IDF + 0.3 × 신경망`처럼 좋은 쪽에 힘을 실으면
손해를 줄일 수 있습니다. 다만 그 가중치도 검증 세트를 보고 정하는 것이라,
01번 12절에서 말한 대로 **검증 세트에 맞춰가는 것**임을 잊지 마세요.

---

## 문제 6. 데이터를 전부 쓰면 신경망이 따라잡나

본문 6절에서 "남은 격차의 이유는 데이터 양"이라고 했습니다. 그 주장을 확인합니다.
학습 데이터를 45,678건 전부로 늘려 **같은 조건에서 두 모델을 다시** 학습합니다.

**이 셀은 2~3분 걸립니다.**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline

전체 = raw[["title", "label"]]

for n in [20_000, len(전체)]:
    d = 전체.sample(n, random_state=RANDOM_STATE) if n < len(전체) else 전체
    xa, xb, ya_t, yb_t = train_test_split(
        d["title"].values, d["label"].values,
        test_size=0.2, stratify=d["label"].values, random_state=RANDOM_STATE,
    )
    le = LabelEncoder()
    ya, yb = le.fit_transform(ya_t), le.transform(yb_t)

    # 신경망 — 본문 6절의 글자 단위 conv
    _, _, nn_acc, nn_time = train_model(
        split="character", seq_len=40, max_tokens=3000, head="conv",
        X_fit=xa, y_fit=ya, X_ev=xb, y_ev=yb,
    )

    # TF-IDF
    tf_acc = accuracy_score(yb_t, make_pipeline(
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)),
        LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    ).fit(xa, ya_t).predict(xb))

    print(f"학습 {len(xa):>6,}건 | 신경망 {nn_acc:.4f} ({nn_time:.0f}초) "
          f"| TF-IDF {tf_acc:.4f} | 격차 {tf_acc - nn_acc:+.4f}")

**신경망이 더 크게 이득을 봤습니다. 하지만 따라잡지는 못했습니다.**

| 학습 데이터 | 신경망 (글자 conv) | TF-IDF | 격차 |
|---|---|---|---|
| 16,000건 | 0.7928 | 0.8455 | **0.053** |
| 36,542건 (2.3배) | 0.8197 | 0.8577 | **0.038** |
| 늘어난 폭 | **+0.027** | +0.012 | −0.015 |

**본문 6절의 주장이 부분적으로 맞았습니다.**

데이터를 2.3배로 늘리자 신경망은 +2.7%p, TF-IDF는 +1.2%p를 얻었습니다.
**신경망이 두 배 넘게 이득을 봤습니다.** 학습할 파라미터가 많은 모델일수록 데이터를 더 갈망한다는
일반적인 성질이 그대로 나타난 것입니다. 격차도 0.053에서 0.038로 좁혀졌습니다.

**그런데 이 속도로는 부족합니다.** 2.3배를 늘려 격차를 0.015 줄였습니다. 남은 0.038을 마저 지우려면
단순히 계산해도 **데이터가 지금의 몇 배는 더 필요합니다.** 그리고 두 곡선 모두 수확 체감 중이라
실제로는 그보다 훨씬 더 들 것입니다.

**결론을 정확히 말하면 이렇습니다.**

> 데이터 양은 격차의 **일부**를 설명합니다. 그러나 "데이터만 더 모으면 신경망이 이긴다"는 말은
> 이 규모에서는 **실용적으로 틀렸습니다.** 이 문제에서 임베딩을 처음부터 학습하는 방식은
> 애초에 TF-IDF보다 비효율적입니다.

**그럼 신경망으로 이기려면?** 임베딩을 **처음부터 학습하지 않으면 됩니다.**
KLUE-RoBERTa 같은 사전 학습 모델은 이미 대량의 한국어로 학습해뒀기 때문에, 우리가 가진 데이터는
"언어를 배우는 데"가 아니라 **"주제를 구분하는 데"만** 쓰입니다.
[KLUE 논문](https://arxiv.org/abs/2105.09680)이 이 데이터셋에 대해 보고한 점수는
macro f1 기준 85 이상으로, 우리가 만든 모델과는 다른 수준입니다.
**같은 데이터인데 격차가 이만큼 나는 것은 모델 구조가 아니라 사전 학습 때문입니다.**

> **이것이 이 시리즈의 마지막 교훈입니다.** "딥러닝이 졌다"의 진짜 의미는
> "딥러닝이 나쁘다"가 아니라 **"이 방식으로 이 규모에서는 진다"** 입니다.
> 그 조건을 정확히 아는 것이, 다음에 무엇을 시도할지 정하는 근거가 됩니다.

---

## 정리

| 문제 | 배운 것 |
|---|---|
| 1 | 시퀀스 길이를 줄이면 **뒤가 잘린다.** 분위수를 보고 정한다 |
| 2 | `max_tokens`를 키워 **OOV를 줄여도 성능은 오르지 않는다.** 한 번 나온 단어는 학습되지 않는다 |
| 3 | 임베딩 차원은 **키운 만큼 돌아오지 않는다.** +0.0067에 학습 시간 3.4배 |
| 4 | 임베딩의 "의미"는 **이 분류 문제에 유용한 정도**일 뿐, 범용 의미 공간이 아니다 |
| 5 | 앙상블의 조건은 서로 다른 실수 **그리고 엇비슷한 실력.** 약한 모델은 발목을 잡는다 |
| 6 | 데이터는 신경망 쪽에 더 이롭지만, **이 규모에서 격차를 지우지는 못한다** |

**전체를 관통하는 것 하나.** 이 여섯 문제에서 손잡이를 여러 개 돌려봤습니다 —
시퀀스 길이, 사전 크기, 임베딩 차원, 앙상블, 데이터 양. 그중 어느 것도
본문 6절에서 **토큰 단위를 바꿔 얻은 +0.044**를 넘지 못했습니다.

**성능이 안 나올 때 손잡이부터 돌리지 마세요.** 모델이 입력을 어떻게 보고 있는지 먼저 확인하면,
돌려야 할 손잡이가 어느 것인지 대개 그 자리에서 드러납니다.